In [25]:
import arxiv
from arxiv import Search, SortCriterion, Client
from typing import List,Dict,Any,Optional
from pydantic import BaseModel,Field
from datetime import datetime
import logging
import itertools
from dataclasses import dataclass
import requests
import PyPDF2
import io


In [3]:
from elasticsearch import Elasticsearch
es_client=Elasticsearch('http://localhost:9200')

In [4]:
logging.basicConfig(level=logging.INFO)
logger=logging.getLogger()

In [5]:
@dataclass
class PaperMetadata:
    paper_id: str
    title: str
    authors: List[str]
    abstract: str
    url: str
    published_date: str

In [8]:

class ArxivSearchTool:
    """Tool for searching ArXiv papers"""
    
    def __init__(self):
        self.name = "search_tool"
        
    def search_arxiv(query: str, max_results: int = 2) -> List[PaperMetadata]:
        """ search arxiv and return papers metadata"""
        logging.info(f"searching arxiv papers for : {query}")
        search = arxiv.Search(
            query = query,
            max_results = max_results,
            sort_by = arxiv.SortCriterion.Relevance
        )
    
        client = Client()
    
        papers = []
        for result in client.results(search):
            paper = PaperMetadata(
                paper_id = result.entry_id.split('/')[-1] ,
                title = result.title,
                authors = [a.name for a in result.authors],
                abstract = result.summary,
                url = result.pdf_url,
                published_date = result.published.strftime("%Y-%m-%d")
            )
            papers.append(paper)
        logging.info(f"found {len(papers)} papers")
        return papers
  
    

In [26]:
class FetchTool:
    """Tool for  searching ArXiv papers also have   downloading and extracting text PDF content and chunk the text"""
    
    def __init__(self):
        self.name = "fetch_tool"
    def search_arxiv(query: str, max_results: int = 2) -> List[PaperMetadata]:
        """ search arxiv and return papers metadata"""
        logging.info(f"searching arxiv papers for : {query}")
        search = arxiv.Search(
            query = query,
            max_results = max_results,
            sort_by = arxiv.SortCriterion.Relevance
        )
    
        client = Client()
    
        papers = []
        for result in client.results(search):
            paper = PaperMetadata(
                paper_id = result.entry_id.split('/')[-1] ,
                title = result.title,
                authors = [a.name for a in result.authors],
                abstract = result.summary,
                url = result.pdf_url,
                published_date = result.published.strftime("%Y-%m-%d")
            )
            papers.append(paper)
        logging.info(f"found {len(papers)} papers")
        return papers
    def extract_text_from_pdf(url: str) -> str:
        """ download pdf and extract text """
        logger.info(f"fetching pdf from url :{url}")
        #download
        response=requests.get(url)
        pdf_file = io.BytesIO(response.content)
        #extract text
        reader = PyPDF2.PdfReader(pdf_file)
        text =""
        for page in reader.pages:
            text += page.extract_text() + "\n"
        return text 
    def chunk_text(text: str, paper_id: str, 
                     chunk_size: int = 1000, overlap: int = 200) ->  List[str]:
        """Chunk text """
        try:
            logger.info(f" Chunking text for paper: {paper_id}")
            
            # Chunk text
            chunks = self._chunk_text(text, paper_id, chunk_size, overlap)
            
            logger.info(f" Created {len(chunks)} chunks")
            
        except Exception as e:
            logger.error(f" Error: {e}")

In [27]:
class BM25SearchTool:
    """
    Tools for BM25 keyword search using Elasticsearch
    """
    def __init__(self,es_client):
        self.name = "BM25_search_tool"
        self.es = es_client
        self.index_name= arxiv_papers
    def index_chunk(self, chunk, paper_metadata:PaperMetadata):
        """Index a chunk for BM25 search"""
        doc = {
            "chunk_id": chunk.chunk_id,
            "paper_id": chunk.paper_id,
            "text": chunk.text,
            "chunk_index": chunk.chunk_index,
            "section": chunk.section,
            "paper_title": paper_metadata.title,
            "authors": paper_metadata.authors,
            "abstract": paper_metadata.abstract,
            "published_date": paper_metadata.published_date
        }
        
        self.es.index(index=self.index_name,id = chunk.chunk_id,document=doc)
    def search(self,query:str,top_k:int =10):
        """Search using BM25"""
        search_body = {
                "query": {
                    "multi_match": {
                        "query": query,
                        "fields": ["text^2", "paper_title^1.5", "abstract"],  # Boost text and title
                        "type": "best_fields"
                    }
                },
                "size": top_k
        }
        response = self.es.search(index=self.index_name, body=search_body)
        # Format results
        results = []
        for hit in response['hits']['hits']:
            results.append(SearchResult(
                paper_id=hit['_source']['paper_id'],
                chunk_id=hit['_source']['chunk_id'],
                text=hit['_source']['text'],
                score=hit['_score'],
                source="bm25",
                metadata={
                    "paper_title": hit['_source']['paper_title'],
                    "chunk_index": hit['_source']['chunk_index']
                }
            ))
        return results
        
        

In [28]:
from agents import Agent, function_tool, Runner
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIAgentsSDKRunner

chat_interface = IPythonChatInterface()

In [32]:
from toyaikit.tools import wrap_instance_methods
search_tools = wrap_instance_methods(FetchTool,BM25SearchTool)

In [33]:
search_instructions = """
You're a helpful assistant that helps user by searching the research article by using tools like fetch tool to retrieve papers related to query and 
find the url and metadata associated with it and retrieve the pdf content from  tools like extract_text_from_pdf and chunk it and index the text and use
BM25SearchTool to find relevant answer to question and while answering pls include chunk_id and paper_id
"""        

In [34]:
search_agent = Agent(
    name='search_agent',
    instructions = search_instructions,
    model='gpt-4o-mini',
    tools=search_tools
)

In [35]:
runner = OpenAIAgentsSDKRunner(
    chat_interface=chat_interface,
    agent=search_agent
)

await runner.run();

You: attention_models


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


You: give summary for Dynamic Attention for Neural Machine Translation


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


...

You: Give the exact formula used in this paper


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


You: stop


Chat ended.


In [21]:
summary_instructions = """
You're a helpful assistant that helps user by summarizing the research article by using tools like search_arxiv to retrieve papers related to query and
use tools like extract_text_from_pdf to get text from research pdf

"""

In [26]:
from toyaikit.tools import wrap_instance_methods
summarize_tools = wrap_instance_methods(search_arxiv,extract_text_from_pdf)

In [27]:
summarize_agent = Agent(
    name='summarize_agent',
    instructions = summary_instructions,
    model='gpt-4o-mini',
    tools=summarize_tools
)

In [28]:
runner = OpenAIAgentsSDKRunner(
    chat_interface=chat_interface,
    agent=summarize_agent
)

await runner.run();

You: attention is all you need


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


You: stop


Chat ended.
